# 03 · Models

Logistic Regression (baseline), Random Forest and a feed-forward Neural Network, fitted
separately in each district. Preprocessing, model settings and validation are described in
`src/modeling.py`; every preprocessing step is learned from the training data only.

**Primary analysis:** school-dependent predictors excluded; 5-fold cross-validation repeated
5 times (25 test sets of 20 %); all children of a household in the same fold.

**Sensitivity analyses** (3 repeats each):

| Analysis | School-dependent predictors | Folds |
|---|---|---|
| primary | excluded | households kept together |
| child_split | excluded | children assigned at random |
| with_school_vars | included | households kept together |
| with_school_vars_child_split | included | children assigned at random |
| without_concurrent | excluded, and the child's own work and computer / mobile use also excluded | households kept together |

**Output:** `results/tables/*.csv`; cross-validation predictions are cached in `results/cv/`.

1. Setup
2. Cross-validation
3. Performance per test set
4. Table 2: best ML model vs logistic regression
5. Table A1: all models and metrics
6. Sensitivity analyses
7. Table B1: top-10 predictors (SHAP)
8. Ranks of predictors discussed in the text

## Step 1 · Setup

In [1]:
import sys
sys.path.insert(0, "..")
import src  # noqa: F401  - loads TensorFlow before pandas (required on Windows)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

import time

import joblib
from scipy.stats import spearmanr

from src import evaluation as ev
from src import modeling
from src.config import (CONCURRENT, DATA_DIR, DISTRICTS, FEATURE_NAMES, FEATURES, LABEL,
                        N_REPEATS, RESULTS_DIR, SCHOOL_DEPENDENT, TARGET)

CV_DIR, TABLES = RESULTS_DIR / "cv", RESULTS_DIR / "tables"
CV_DIR.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

data = pd.read_parquet(DATA_DIR / "analysis_data.parquet")
WITHOUT_SCHOOL = [f for f in FEATURES if f not in SCHOOL_DEPENDENT]
ORDER = list(DISTRICTS)
ANALYSES = {
    "primary":                      dict(features=WITHOUT_SCHOOL, household=True, repeats=N_REPEATS),
    "child_split":                  dict(features=WITHOUT_SCHOOL, household=False, repeats=3),
    "with_school_vars":             dict(features=FEATURES, household=True, repeats=3),
    "with_school_vars_child_split": dict(features=FEATURES, household=False, repeats=3),
    "without_concurrent":           dict(features=[f for f in WITHOUT_SCHOOL if f not in CONCURRENT],
                                         household=True, repeats=3),
}
pd.DataFrame({k: {"predictors": len(v["features"]), "households together": v["household"],
                  "repeats": v["repeats"]} for k, v in ANALYSES.items()}).T

,predictors,households together,repeats
primary,53,True,5
child_split,53,False,3
with_school_vars,56,True,3
with_school_vars_child_split,56,False,3
without_concurrent,40,True,3


## Step 2 · Cross-validation
Each analysis and district is fitted once and cached. SHAP values are computed in the primary analysis, first repeat: every child is explained once, by a model that did not see that child's household.

In [2]:
def district_data(key):
    return data.query("district_key == @key").reset_index(drop=True)

def cross_validate(analysis, key):
    path = CV_DIR / f"{analysis}_{key}.joblib"
    if not path.exists():
        a = ANALYSES[analysis]
        joblib.dump(modeling.run_cv(district_data(key), a["features"], TARGET,
                                    household=a["household"], n_repeats=a["repeats"],
                                    explain_first_repeat=(analysis == "primary")), path)
    return joblib.load(path)

CV, timing = {}, {}
for analysis in ANALYSES:
    for key in ORDER:
        t = time.time()
        CV[analysis, key] = cross_validate(analysis, key)
        timing[analysis, key] = round(time.time() - t, 1)
pd.Series(timing).unstack(0).rename(index=LABEL).rename_axis("seconds", axis=1)

seconds,child_split,primary,with_school_vars,with_school_vars_child_split,without_concurrent
Dera Bugti,61.2,70.3,22.6,17.6,17.8
Islamabad,46.4,79.8,63.0,28.3,20.5
Karachi Central,53.0,85.5,39.9,32.6,23.4
Kohistan,71.2,96.8,28.1,22.3,22.4
Lahore,84.1,332.6,108.9,49.6,35.4
Peshawar,87.2,200.4,94.7,36.6,31.9
Quetta,57.2,109.7,86.4,32.8,24.6
Rajanpur,83.7,109.7,27.2,22.9,23.4
Tharparkar,50.8,79.9,20.3,20.1,20.5


## Step 3 · Performance per test set
Every model's metrics on every held-out test set; mean AUC per analysis.

In [3]:
folds = pd.concat([ev.summarise(CV[a, k], district_data(k), TARGET).assign(analysis=a, district=k)
                   for a, k in CV], ignore_index=True)
folds.to_csv(TABLES / "all_test_sets.csv", index=False)

def ci(analysis, key, model, metric):
    d = folds.query("analysis == @analysis and district == @key and model == @model")
    return ev.corrected_ttest(d[metric], d["n_train"].mean(), d["n_test"].mean())

folds.groupby(["analysis", "model"])["auc"].mean().unstack().loc[list(ANALYSES)].round(3)

model,LR,NN,RF
analysis,,,
primary,0.768,0.774,0.828
child_split,0.791,0.824,0.863
with_school_vars,0.777,0.786,0.837
with_school_vars_child_split,0.798,0.830,0.869
without_concurrent,0.742,0.753,0.809


## Step 4 · Table 2: best ML model vs logistic regression (primary analysis)
- AUC and 95 % CI: mean over the 25 test sets, corrected resampled t-test (Nadeau & Bengio, 2003).
- Difference: paired over the same 25 test sets, corrected resampled t-test.
- DeLong test: both models' out-of-fold predictions for every child (first repeat); assumes children are independent.
- Household-cluster bootstrap: 2,000 resamples of whole households, so siblings stay together.
- p-values Holm-adjusted across the nine districts.

In [4]:
rows = []
for key in ORDER:
    d = folds.query("analysis == 'primary' and district == @key")
    best = d.groupby("model")["auc"].mean()[["RF", "NN"]].idxmax()
    paired = d.pivot_table(index=["repeat", "fold"], columns="model", values="auc")
    diff = ev.corrected_ttest(paired[best] - paired["LR"], d["n_train"].mean(), d["n_test"].mean())
    dk = district_data(key)
    oof = ev.out_of_fold(CV["primary", key], len(dk))
    y, groups = dk[TARGET].to_numpy(), dk["hhcode"].to_numpy()
    dl = ev.delong(y, oof[best].to_numpy(), oof["LR"].to_numpy())
    cb = ev.cluster_bootstrap_auc_diff(y, oof[best].to_numpy(), oof["LR"].to_numpy(), groups)
    lr, ml = ci("primary", key, "LR", "auc"), ci("primary", key, best, "auc")
    rows.append({
        "District type": DISTRICTS[key][2], "District": LABEL[key], "MPI": DISTRICTS[key][3],
        "Baseline AUC (LR)": lr["mean"], "LR 95% CI": f"[{lr['ci_lo']:.3f}, {lr['ci_hi']:.3f}]",
        "Best ML model": modeling.MODEL_NAMES[best],
        "Best ML AUC": ml["mean"], "ML 95% CI": f"[{ml['ci_lo']:.3f}, {ml['ci_hi']:.3f}]",
        "Difference (AUC)": diff["mean"],
        "Difference 95% CI": f"[{diff['ci_lo']:+.3f}, {diff['ci_hi']:+.3f}]",
        "Relative lift %": diff["mean"] / lr["mean"] * 100,
        "p corrected t": diff["p"], "p DeLong": dl["p"], "p cluster bootstrap": cb["p"],
        "Cluster bootstrap 95% CI": f"[{cb['ci_lo']:+.3f}, {cb['ci_hi']:+.3f}]",
    })
table2 = pd.DataFrame(rows)
for test in ["p corrected t", "p DeLong", "p cluster bootstrap"]:
    table2[f"{test} (Holm)"] = ev.holm(table2[test])
table2["Significant (all three tests)"] = np.where(
    table2[["p corrected t (Holm)", "p DeLong (Holm)", "p cluster bootstrap (Holm)"]].max(axis=1)
    < 0.05, "Yes", "No")
table2.to_csv(TABLES / "table2_performance.csv", index=False)
table2.round(4)

,District type,District,MPI,Baseline AUC (LR),LR 95% CI,Best ML model,Best ML AUC,ML 95% CI,Difference (AUC),Difference 95% CI,Relative lift %,p corrected t,p DeLong,p cluster bootstrap,Cluster bootstrap 95% CI,p corrected t (Holm),p DeLong (Holm),p cluster bootstrap (Holm),Significant (all three tests)
0,Federal and Provincial Capitals,Karachi Central,0.02,0.7609,"[0.741, 0.781]",Random Forest,0.8066,"[0.785, 0.828]",0.0458,"[+0.023, +0.069]",6.0156,0.0004,0.0,0.0005,"[+0.016, +0.056]",0.0013,0.0,0.0045,Yes
1,Federal and Provincial Capitals,Islamabad,0.01,0.7363,"[0.669, 0.804]",Random Forest,0.8143,"[0.756, 0.873]",0.0781,"[+0.021, +0.135]",10.6022,0.0090,0.0,0.0005,"[+0.031, +0.113]",0.0112,0.0,0.0045,Yes
2,Federal and Provincial Capitals,Lahore,0.03,0.7806,"[0.760, 0.801]",Random Forest,0.8157,"[0.796, 0.835]",0.0351,"[+0.018, +0.052]",4.5018,0.0002,0.0,0.0005,"[+0.024, +0.046]",0.0009,0.0,0.0045,Yes
3,Federal and Provincial Capitals,Quetta,0.20,0.7526,"[0.729, 0.776]",Random Forest,0.8109,"[0.784, 0.838]",0.0583,"[+0.034, +0.083]",7.7485,0.0001,0.0,0.0005,"[+0.049, +0.080]",0.0003,0.0,0.0045,Yes
4,Federal and Provincial Capitals,Peshawar,0.11,0.7615,"[0.737, 0.786]",Random Forest,0.8247,"[0.803, 0.846]",0.0632,"[+0.046, +0.081]",8.2950,0.0000,0.0,0.0005,"[+0.046, +0.067]",0.0000,0.0,0.0045,Yes
5,High-Deprivation,Kohistan,0.53,0.8467,"[0.817, 0.876]",Random Forest,0.9089,"[0.888, 0.930]",0.0622,"[+0.049, +0.075]",7.3401,0.0000,0.0,0.0005,"[+0.048, +0.072]",0.0000,0.0,0.0045,Yes
6,High-Deprivation,Dera Bugti,0.36,0.7596,"[0.709, 0.810]",Random Forest,0.8767,"[0.848, 0.905]",0.1171,"[+0.074, +0.160]",15.4207,0.0000,0.0,0.0005,"[+0.095, +0.163]",0.0000,0.0,0.0045,Yes
7,High-Deprivation,Rajanpur,0.32,0.8007,"[0.779, 0.823]",Random Forest,0.8268,"[0.805, 0.849]",0.0261,"[+0.008, +0.044]",3.2608,0.0056,0.0,0.0005,"[+0.013, +0.040]",0.0112,0.0,0.0045,Yes
8,High-Deprivation,Tharparkar,0.51,0.7126,"[0.685, 0.740]",Random Forest,0.7671,"[0.735, 0.799]",0.0545,"[+0.037, +0.072]",7.6512,0.0000,0.0,0.0005,"[+0.041, +0.080]",0.0000,0.0,0.0045,Yes


## Step 5 · Table A1: all models and metrics (primary analysis)
Mean over the 25 test sets. Out-of-school class: precision, recall, F1 and PR-AUC (threshold 0.5 for precision, recall and F1). Survey-weighted AUC uses the PSLM sampling weights.

In [5]:
metrics = ["accuracy", "precision_out", "recall_out", "f1_out", "pr_auc_out", "f1_attending",
           "auc", "auc_weighted", "prevalence_out"]
a1 = folds.query("analysis == 'primary'").groupby(["district", "model"])[metrics].mean().reset_index()
a1["AUC 95% CI"] = [f"[{c['ci_lo']:.3f}, {c['ci_hi']:.3f}]"
                    for c in (ci("primary", r.district, r.model, "auc") for r in a1.itertuples())]
counts = data.groupby("district_key")[TARGET].agg(children="size", attending="sum")
a1["children"] = a1["district"].map(counts["children"])
a1["out of school (n)"] = a1["district"].map(counts["children"] - counts["attending"])
a1["district"] = pd.Categorical(a1["district"].map(LABEL), [LABEL[k] for k in ORDER])
a1["model"] = a1["model"].map(modeling.MODEL_NAMES)
a1 = a1.sort_values(["district", "model"])
a1.to_csv(TABLES / "tableA1_all_metrics.csv", index=False)
a1.round(3)

,district,model,accuracy,precision_out,recall_out,f1_out,pr_auc_out,f1_attending,auc,auc_weighted,prevalence_out,AUC 95% CI,children,out of school (n)
6,Karachi Central,Logistic Regression,0.714,0.405,0.663,0.502,0.497,0.799,0.761,0.754,0.218,"[0.741, 0.781]",2917,636
7,Karachi Central,Neural Network,0.784,0.507,0.509,0.505,0.516,0.861,0.764,0.759,0.218,"[0.735, 0.793]",2917,636
8,Karachi Central,Random Forest,0.802,0.558,0.474,0.509,0.564,0.876,0.807,0.795,0.218,"[0.785, 0.828]",2917,636
3,Islamabad,Logistic Regression,0.783,0.263,0.577,0.355,0.361,0.869,0.736,0.745,0.105,"[0.669, 0.804]",1696,179
4,Islamabad,Neural Network,0.866,0.375,0.369,0.359,0.369,0.925,0.747,0.755,0.105,"[0.677, 0.818]",1696,179
5,Islamabad,Random Forest,0.900,0.570,0.183,0.266,0.402,0.946,0.814,0.813,0.105,"[0.756, 0.873]",1696,179
12,Lahore,Logistic Regression,0.749,0.353,0.653,0.457,0.511,0.837,0.781,0.775,0.162,"[0.760, 0.801]",7956,1288
13,Lahore,Neural Network,0.810,0.424,0.469,0.443,0.477,0.885,0.758,0.752,0.162,"[0.734, 0.782]",7956,1288
14,Lahore,Random Forest,0.858,0.590,0.403,0.477,0.552,0.918,0.816,0.811,0.162,"[0.796, 0.835]",7956,1288
18,Quetta,Logistic Regression,0.690,0.541,0.689,0.606,0.631,0.743,0.753,0.744,0.348,"[0.729, 0.776]",3259,1134


## Step 6 · Sensitivity analyses
Mean AUC (nine districts) under each analysis, and the best ML model's AUC per district.

In [6]:
summary = folds.groupby(["analysis", "model"])["auc"].mean().unstack().loc[list(ANALYSES)]
summary["RF - LR"] = summary["RF"] - summary["LR"]
summary.round(3).to_csv(TABLES / "sensitivity_summary.csv")

by_district = folds.groupby(["district", "analysis", "model"])["auc"].mean().unstack("model")
by_district = by_district[["RF", "NN"]].max(axis=1).unstack("analysis")[list(ANALYSES)]
by_district.index = by_district.index.map(LABEL)
by_district = by_district.loc[[LABEL[k] for k in ORDER]]
by_district.round(3).to_csv(TABLES / "sensitivity_best_ml_by_district.csv")
display(summary.round(3))
by_district.round(3)

model,LR,NN,RF,RF - LR
analysis,,,,
primary,0.768,0.774,0.828,0.060
child_split,0.791,0.824,0.863,0.072
with_school_vars,0.777,0.786,0.837,0.060
with_school_vars_child_split,0.798,0.830,0.869,0.071
without_concurrent,0.742,0.753,0.809,0.067


analysis,primary,child_split,with_school_vars,with_school_vars_child_split,without_concurrent
district,,,,,
Karachi Central,0.807,0.852,0.819,0.860,0.783
Islamabad,0.814,0.868,0.819,0.873,0.782
Lahore,0.816,0.853,0.833,0.860,0.779
Quetta,0.811,0.846,0.836,0.864,0.781
Peshawar,0.825,0.857,0.841,0.870,0.790
Kohistan,0.909,0.927,0.909,0.927,0.909
Dera Bugti,0.877,0.901,0.877,0.901,0.872
Rajanpur,0.827,0.855,0.829,0.855,0.826
Tharparkar,0.767,0.809,0.767,0.809,0.761


## Step 7 · Table B1: top-10 predictors (Random Forest SHAP, primary analysis)
Mean absolute SHAP value over all children of the district (out-of-fold), with indicator columns summed back to their predictor. The last column is the rank agreement with logistic-regression SHAP values.

In [7]:
top10, agreement = {}, {}
for key in ORDER:
    first_repeat = [r for r in CV["primary", key] if r["repeat"] == 0]
    rf = pd.concat([r["shap"]["RF"] for r in first_repeat]).fillna(0).abs().mean()
    lr = pd.concat([r["shap"]["LR"] for r in first_repeat]).fillna(0).abs().mean()
    rf = rf.sort_values(ascending=False)
    top10[LABEL[key]] = [FEATURE_NAMES[v] for v in rf.index[:10]]
    shared = rf.index.intersection(lr.index)
    agreement[LABEL[key]] = spearmanr(rf[shared], lr[shared]).correlation

table_b1 = pd.DataFrame(top10, index=[f"#{i}" for i in range(1, 11)]).T
table_b1["#9-10"] = table_b1["#9"] + "; " + table_b1["#10"]
table_b1 = table_b1.drop(columns=["#9", "#10"])
table_b1["RF-LR rank agreement (Spearman)"] = pd.Series(agreement).round(2)
table_b1.to_csv(TABLES / "tableB1_top_predictors.csv", index_label="District")
table_b1

,#1,#2,#3,#4,#5,#6,#7,#8,#9-10,RF-LR rank agreement (Spearman)
Karachi Central,Head education level,Has internet,Pay for water,Age,No-computer reason,Worried about food,Dwelling type,Used mobile,Head age; Used computer,0.59
Islamabad,Head education level,Has internet,Used mobile,Age,Household income (log),Occupancy status,Head age,No-computer reason,Worried about food; Connected to sewerage,0.50
Lahore,Head education level,Age,Has internet,Worried about food,Used mobile,Worked last month,Occupancy status,Has handwashing place,Income used for household; Total assets,0.53
Quetta,Gender,Toilet type,Age,Head education level,Has handwashing place,Has internet,Reason for headship,Used computer,Used mobile; No-computer reason,0.76
Peshawar,Gender,Age,Head education level,Used mobile,Region (urban/rural),Has handwashing place,Worked last month,Used computer,Income used for household; Has internet,0.67
Kohistan,Gender,Age,Heating fuel,Head education level,Handwashing water source,Cooking water source,Toilet type,Has internet,Household income (log); Number of rooms,0.80
Dera Bugti,Age,Gender,Total assets,Used mobile,Head education level,Household income (log),Cooking water source,Toilet type,Heating fuel; Head age,0.82
Rajanpur,Toilet type,Head education level,Gender,Connected to sewerage,Handwashing water source,Region (urban/rural),Used mobile,Shared toilet,Head age; Heating fuel,0.81
Tharparkar,Gender,Head education level,Age,Used mobile,Shared toilet,Total assets,Toilet type,Head age,Household income (log); Number of rooms,0.83


## Step 8 · Ranks of predictors discussed in the text
Rank of each predictor by mean absolute Random Forest SHAP value (as in Table B1), and the mean SHAP value of gender for girls (negative = girls have a lower predicted probability of attending).

In [8]:
SELECTED = ["head_edu_level_num", "age", "gender", "has_internet", "toilet_type", "disability"]
ranks, girls = {}, {}
for key in ORDER:
    first_repeat = [r for r in CV["primary", key] if r["repeat"] == 0]
    shap_rf = pd.concat([r["shap"]["RF"] for r in first_repeat]).fillna(0)
    order = shap_rf.abs().mean().sort_values(ascending=False).index.tolist()
    ranks[LABEL[key]] = {FEATURE_NAMES[f]: order.index(f) + 1 if f in order else None for f in SELECTED}
    is_girl = district_data(key).loc[shap_rf.index, "gender"].astype(str) == "female"
    girls[LABEL[key]] = shap_rf.loc[is_girl, "gender"].mean() if "gender" in shap_rf else np.nan
predictor_ranks = pd.DataFrame(ranks).T
predictor_ranks["Mean SHAP of gender for girls"] = pd.Series(girls).round(3)
predictor_ranks.to_csv(TABLES / "predictor_ranks.csv", index_label="District")
predictor_ranks

,Head education level,Age,Gender,Has internet,Toilet type,Disability,Mean SHAP of gender for girls
Karachi Central,1,4,36,2,34,33,0.000
Islamabad,1,4,23,2,17,19,0.004
Lahore,1,2,23,3,28,15,0.007
Quetta,4,3,1,6,2,34,-0.074
Peshawar,3,2,1,10,18,24,-0.106
Kohistan,4,2,1,8,7,31,-0.329
Dera Bugti,5,1,2,20,8,36,-0.095
Rajanpur,2,19,3,28,1,20,-0.056
Tharparkar,2,3,1,13,7,21,-0.105
